In [6]:

# pip install faiss-cpu

In [10]:

from google import genai

import os

from dotenv import load_dotenv

In [11]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [12]:

import faiss

import numpy as np

In [13]:

docs = [
    "Our company policy states that all employees must complete security training annually.",
    "The employee onboarding process takes approximately two weeks.",
    "You can reset your company email password via the IT support portal.",
    "Remote work is allowed up to 3 days per week with manager approval."
]


In [14]:

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=docs
)


In [15]:

doc_embeddings = np.array(
    [embedding.values for embedding in result.embeddings],
    dtype="float32"
)

print("Embedding Shape:", doc_embeddings.shape)


Embedding Shape: (4, 3072)


In [16]:

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings)

print("Documents stored in FAISS:", index.ntotal)


Documents stored in FAISS: 4


In [17]:

def answer_question(question):

    # Convert question into Gemini embedding

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[question]
    )

    q_embedding = np.array(
        [result.embeddings[0].values],
        dtype="float32"
    )


    # Search FAISS

    D, I = index.search(q_embedding, k=2)


    # Retrieve documents

    retrieved_docs = [docs[i] for i in I[0]]


    print("\n========== RETRIEVED DOCUMENTS ==========")

    for i, doc in enumerate(retrieved_docs, 1):
        print(f"{i}. {doc}")


    # Create context

    context = "\n\n".join(retrieved_docs)


    # Create RAG prompt

    prompt = f"""
You are a helpful company assistant.

Answer the question ONLY using the context below.

If the answer is not available in the context,
say "I don't know based on the provided documents."

CONTEXT:
{context}

QUESTION:
{question}

Give a concise and accurate answer.
"""


    # Gemini generates final answer

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text


In [18]:
# ASK QUESTION

question = "How do I reset my email password?"

answer = answer_question(question)


========== RETRIEVED DOCUMENTS ==========
1. You can reset your company email password via the IT support portal.
2. Our company policy states that all employees must complete security training annually.


In [19]:

print(answer)

You can reset your company email password via the IT support portal.
